<a href="https://colab.research.google.com/github/sangjkim930/Project-Based-Learning/blob/main/Week_03_Gemini_API/Week3_Gemini_API_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Week 3 Exercise: Evidence-Based Research with Gemini API
# ============================================================

# Install the Google GenAI SDK.
# Colab may already have this package installed.

!pip install -q google-genai

In [2]:
# Import the libraries needed for this exercise.

from google import genai
from google.genai import types
from google.colab import userdata, files

In [ ]:
# ============================================================
# Connect to the Gemini API
# ============================================================

# Load the API key securely from Colab Secrets.
# The secret must be saved with the name:
# GEMINI_API_KEY

api_key = userdata.get("Gemini_API_Key")

# Create the Gemini API client.
client = genai.Client(api_key=api_key)

print("Gemini API connection is ready.")

In [ ]:
# ============================================================
# Select the Gemini model
# ============================================================

MODEL_NAME = "gemini-3.8-flash"

print("Model:", MODEL_NAME)

In [ ]:
# ============================================================
# Upload the three Nepal reports to Google Colab
# ============================================================

print("Please select the THREE PDF files provided for this exercise.")

uploaded_files = files.upload()

print("\nFiles uploaded to Colab:")
for filename in uploaded_files.keys():
    print("-", filename)

In [ ]:
# ============================================================
# Check whether the correct files were uploaded
# ============================================================

pdf_files = [
    "01_Nepal_Economy_2024_25_NRB.pdf",
    "02_Nepal_Development_Strategy_2025_29_ADB.pdf",
    "03_Nepal_Development_Update_2026_WorldBank.pdf"
]

missing_files = [
    filename for filename in pdf_files
    if filename not in uploaded_files
]

if len(missing_files) == 0:
    print("All three required files are ready.")
else:
    print("The following files are missing:")
    for filename in missing_files:
        print("-", filename)

In [ ]:
# ============================================================
# Upload the PDF files to Gemini
# ============================================================

gemini_docs = []

print("Uploading documents to Gemini...\n")

for file_path in pdf_files:

    uploaded = client.files.upload(file=file_path)

    gemini_docs.append(uploaded)

    print("Uploaded:", uploaded.display_name)

print("\nAll documents have been uploaded to Gemini.")

In [9]:
# ============================================================
# Define evidence-based research instructions
# ============================================================

system_instruction = """
You are an evidence-based research assistant.

Use ONLY the three documents provided in this exercise.

For every important factual claim:

1. Identify the source document.
2. Provide the relevant section when it can be identified.
3. Provide a page number only when it can be clearly verified.
4. Do not invent page numbers, quotations, or citations.
5. If the documents do not provide enough evidence,
   clearly state that the evidence is not available.

Clearly distinguish information from different reports.
"""

In [12]:
# ============================================================
# Create a reusable Gemini research function
# with automatic retry
# ============================================================

import time
import random

def ask_gemini(question, max_retries=5):

    for attempt in range(max_retries):

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,

                contents=[
                    "SOURCE 1: Nepal Rastra Bank Annual Report 2024/25",
                    gemini_docs[0],

                    "SOURCE 2: ADB Nepal Country Partnership Strategy 2025-2029",
                    gemini_docs[1],

                    "SOURCE 3: World Bank Nepal Development Update 2026",
                    gemini_docs[2],

                    question
                ],

                config=types.GenerateContentConfig(
                    system_instruction=system_instruction
                )
            )

            return response.text

        except Exception as e:

            # Wait longer after each failed attempt.
            wait_time = (2 ** attempt) + random.random()

            print(f"Request failed. Retrying in {wait_time:.1f} seconds...")
            time.sleep(wait_time)

    return "The request could not be completed after several attempts. Please try again later."

In [ ]:
# ============================================================
# Query 1: Major development challenges
# ============================================================

question_1 = """
What are the three most important economic and development
challenges facing Nepal?

Support each challenge with evidence from the provided documents.

Present your answer in a table with the following columns:

Challenge | Explanation | Evidence | Source | Page or Section
"""

answer_1 = ask_gemini(question_1)

print(answer_1)

In [ ]:
# ============================================================
# Query 2: Development opportunities
# ============================================================

question_2 = """
What are the three most promising opportunities
for Nepal's future economic and social development?

Use only the provided documents.

For each opportunity, explain why it is important
and identify the supporting source.

Present the results in a clear table.
"""

answer_2 = ask_gemini(question_2)

print(answer_2)

In [ ]:
# ============================================================
# Query 3: Compare perspectives
# ============================================================

question_3 = """
How do the Nepal Rastra Bank, Asian Development Bank,
and World Bank reports differ in their perspectives
on Nepal's economy and development?

For each report, identify:

1. Its main focus
2. Major issues emphasized
3. Important development priorities

Present the comparison in a table.
"""

answer_3 = ask_gemini(question_3)

print(answer_3)

In [ ]:
# ============================================================
# YOUR TURN
# ============================================================

# Change the question below.
#
# Possible topics:
# - Infrastructure
# - Private-sector development
# - Youth employment
# - Digital technology
# - Tourism
# - Remittances
# - Foreign investment
#
# Write your own research question.

my_question = """
What role could digital technology play
in Nepal's future economic development?

Use evidence from the provided documents.
"""

my_answer = ask_gemini(my_question)

print(my_answer)